## Task 1



> Download data from Don’tGetKicked competition.Design train/validation/test split.


> Use the "PurchDate" field to split, test must be later in time than validation, same goes for validation and train: train.PurchDate < valid.PurchDate < test.PurchDate.


> Use the first 33% of the data for the training, the last 33% of the data for the test, and the middle 33% for the validation set. Don't use the test dataset until the end!

> Use LabelEncoder or OneHotEncoder from sklearn to preprocess categorical variables. Be careful with data leakage (fit Encoder to training and apply to validation & test). Consider another coding approach if you encounter new categorical values in validation & test (not seen in training), for example: https://contrib.scikit-learn.org/category_encoders/count.html












In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import numpy as np
from collections import Counter
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

First, I am loading and sorting data

In [2]:
df = pd.read_csv('training.csv')
df['PurchDate'] = pd.to_datetime(df['PurchDate'])
df = df.sort_values(by='PurchDate').reset_index(drop=True)

In [3]:
df.head()


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,8884,0,2009-01-05,MANHEIM,2005,4,CHEVROLET,IMPALA 3.4L V6 SFI,Bas,4D SEDAN,...,5597.0,6826.0,NaN,NaN,16044.0,27542.0,NC,5600.0,0.0,2152.0
1,8885,0,2009-01-05,MANHEIM,2006,3,DODGE,CARAVAN FWD V6 3.3L,SXT,MINIVAN 3.3L SXT,...,7350.0,9023.0,NaN,NaN,17675.0,27542.0,NC,5100.0,0.0,1623.0
2,8882,0,2009-01-05,MANHEIM,2004,5,DODGE,STRATUS 4C 2.4L I4 M,SXT,4D SEDAN SE,...,4126.0,5210.0,NaN,NaN,17675.0,27542.0,NC,4700.0,0.0,1020.0
3,8881,0,2009-01-05,MANHEIM,2004,5,DODGE,DURANGO 4WD V8 4.7L,SLT,4D SUV 4.7L,...,9458.0,10718.0,NaN,NaN,16044.0,27542.0,NC,7600.0,0.0,1411.0
4,8880,1,2009-01-05,MANHEIM,2006,3,FORD,FREESTYLE AWD V6 3.0,SE,4D CUV 3.0L SE,...,7589.0,9522.0,NaN,NaN,16044.0,27542.0,NC,6700.0,0.0,1623.0


defining the target and features to drop

In [4]:
TARGET_COL = 'IsBadBuy'
COLS_TO_DROP_PRE_ENCODE = ['RefId', 'VIN', 'VNZIP1', 'PurchDate']

Defining tge split indices

In [5]:
N = len(df)
train_end_idx = int(N * 0.33)
valid_end_idx = int(N * 0.66)

I am creating time based splits

In [6]:
df_train = df.iloc[:train_end_idx].copy()
df_valid = df.iloc[train_end_idx:valid_end_idx].copy()
df_test = df.iloc[valid_end_idx:].copy()

In [7]:
print(f"Total Rows: {N}")
print(f"Train Rows: {len(df_train)} (0 to {train_end_idx-1})")
print(f"Valid Rows: {len(df_valid)} ({train_end_idx} to {valid_end_idx-1})")
print(f"Test Rows: {len(df_test)} ({valid_end_idx} to {N-1})")

Total Rows: 15912
Train Rows: 5250 (0 to 5249)
Valid Rows: 5251 (5250 to 10500)
Test Rows: 5411 (10501 to 15911)


separating x (features) and y (target)

In [8]:
y_train = df_train[TARGET_COL].values
y_valid = df_valid[TARGET_COL].values
y_test = df_test[TARGET_COL].values

X_train_features = df_train.drop(columns=[TARGET_COL] + COLS_TO_DROP_PRE_ENCODE, errors='ignore')
X_valid_features = df_valid.drop(columns=[TARGET_COL] + COLS_TO_DROP_PRE_ENCODE, errors='ignore')
X_test_features = df_test.drop(columns=[TARGET_COL] + COLS_TO_DROP_PRE_ENCODE, errors='ignore')

Identifying categorical features

In [9]:
CATEGORICAL_FEATURES = [
    'Make', 'Model', 'Trim', 'SubModel', 'Color', 'Transmission',
    'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName',
    'PRIMEUNIT', 'AUCGUART', 'WheelTypeID'
]

identifying numerical features

In [10]:
NUMERICAL_FEATURES = X_train_features.select_dtypes(include=np.number).columns.tolist()
NUMERICAL_FEATURES = [f for f in NUMERICAL_FEATURES if f not in CATEGORICAL_FEATURES]

creating a pipeline for categorical features

In [11]:
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

and pipeline for numerical features

In [12]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

and now im combining the two pipelines usign the column transformer

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, NUMERICAL_FEATURES),
        ('cat', categorical_pipeline, CATEGORICAL_FEATURES)
    ],
    remainder='drop',
    verbose_feature_names_out=False
)

fitting to training data only and transforming all three sets

In [14]:
for col in CATEGORICAL_FEATURES:
    if col in X_train_features.columns:
        X_train_features[col] = X_train_features[col].astype(str)
        X_valid_features[col] = X_valid_features[col].astype(str)
        X_test_features[col] = X_test_features[col].astype(str)

preprocessor.fit(X_train_features)

X_train_processed = preprocessor.transform(X_train_features)
X_valid_processed = preprocessor.transform(X_valid_features)
X_test_processed = preprocessor.transform(X_test_features)

In [15]:
print(f"X_train_processed shape: {X_train_processed.shape}")
print(f"X_valid_processed shape: {X_valid_processed.shape}")

X_train_processed shape: (5250, 1096)
X_valid_processed shape: (5251, 1096)


## Task 2




> Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss).




> It should support fit, predict_proba, and predict methods. Also, the maximum depth (max_depth) must be a parameter of your class. Use the Gini impurity criterion as a criterion for choosing the split.


Here is the blueprint:



```
model = DecisionTreeClassifier(max_depth=7)
model.fit(Xtrain, ytrain)
model.predict_proba(Xvalid)
```



Create a separate class for Node. It should be able to hold data (sample features and targets), compute Gini impurity, and have pointers to children (left and right nodes). For the Regressor, use standard deviation instead of Gini impurity.
Implement a function that finds the best possible split in the current node.
Combine the previous steps into your working Decision Tree Classifier.
Implement an Extra Randomized Tree by designing another function to find the best split.



So heres my class for Node

In [16]:
from collections import Counter
import numpy as np

class _Node:
    """Лёгкий узел без хранения данных."""
    def __init__(self, prediction, depth):
        self.prediction = prediction
        self.depth = depth
        self.feature_index = None
        self.threshold = None
        self.left = None
        self.right = None


And DTC

this class implements the full tree sctructure using the gini impurity criterion

In [17]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=None, extra_randomization=False):
        self.max_depth = max_depth
        self.extra_randomization = extra_randomization
        self.root = None

    # --- impurity ---
    def _gini(self, y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1 - np.sum(p*p)

    # --- find best split ---
    def _find_best_split(self, X, y):
        n_samples, n_features = X.shape
        parent_impurity = self._gini(y)

        best_gain = 0.0
        best_feature, best_thresh = None, None

        if self.extra_randomization:
            features = np.random.choice(n_features, int(np.sqrt(n_features)), replace=False)
        else:
            features = range(n_features)

        for feature in features:
            values = X[:, feature]
            thresholds = np.unique(values)

            for t in thresholds:
                left_mask = values <= t
                if not left_mask.any() or left_mask.all():
                    continue

                y_left = y[left_mask]
                y_right = y[~left_mask]

                w_left = len(y_left) / n_samples
                w_right = 1 - w_left

                gain = parent_impurity - (
                    w_left * self._gini(y_left) +
                    w_right * self._gini(y_right)
                )

                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_thresh = t

        return best_feature, best_thresh, best_gain

    # --- grow tree ---
    def _grow(self, X, y, depth):
        prediction = Counter(y).most_common(1)[0][0]
        node = _Node(prediction=prediction, depth=depth)

        if self.max_depth is not None and depth >= self.max_depth:
            return node
        if len(np.unique(y)) == 1:
            return node
        if len(y) < 2:
            return node

        f, t, gain = self._find_best_split(X, y)
        if gain == 0 or f is None:
            return node

        node.feature_index = f
        node.threshold = t

        mask = X[:, f] <= t
        node.left = self._grow(X[mask], y[mask], depth+1)
        node.right = self._grow(X[~mask], y[~mask], depth+1)

        return node

    # --- fit ---
    def fit(self, X, y):
        self.root = self._grow(X, y, 0)

    # --- predict one ---
    def _predict_one(self, x, node):
        if node.left is None:
            return node.prediction
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    # --- predict ---
    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

    # --- predict_proba ---
    def predict_proba(self, X):
        preds = self.predict(X)
        classes = np.unique(preds)
        m = len(X)
        proba = np.zeros((m, 2))

        for i, p in enumerate(preds):
            if p == 0:
                proba[i] = [1, 0]
            else:
                proba[i] = [0, 1]
        return proba


DTR

In [18]:
class DecisionTreeRegressor:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    def _mse(self, y):
        return np.var(y)

    def _find_best_split(self, X, y):
        n_samples, n_features = X.shape
        parent = self._mse(y)

        best_gain = 0
        best_feature = None
        best_thresh = None

        for f in range(n_features):
            values = X[:, f]
            thresholds = np.unique(values)

            for t in thresholds:
                mask = values <= t
                if not mask.any() or mask.all():
                    continue

                y_left = y[mask]
                y_right = y[~mask]

                gain = parent - (
                    len(y_left)/n_samples * self._mse(y_left) +
                    len(y_right)/n_samples * self._mse(y_right)
                )

                if gain > best_gain:
                    best_gain = gain
                    best_feature = f
                    best_thresh = t

        return best_feature, best_thresh, best_gain

    def _grow(self, X, y, depth):
        prediction = np.mean(y)
        node = _Node(prediction, depth)

        if self.max_depth is not None and depth >= self.max_depth:
            return node

        f, t, gain = self._find_best_split(X, y)
        if gain == 0 or f is None:
            return node

        node.feature_index = f
        node.threshold = t
        mask = X[:, f] <= t

        node.left = self._grow(X[mask], y[mask], depth+1)
        node.right = self._grow(X[~mask], y[~mask], depth+1)

        return node

    def fit(self, X, y):
        self.root = self._grow(X, y, 0)

    def _predict_one(self, x, node):
        if node.left is None:
            return node.prediction
        if x[node.feature_index] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])


Extra Randomized Tree Functionality

The function to implement an Extra Randomized Tree (Extremely Randomized Tree) has been integrated into the _find_best_split method of both classes using the randomized=True parameter.

## Task 3

Since my model is custom, systematically test a range of 'max_depth' values on the validation set

In [19]:
from sklearn.metrics import roc_auc_score
max_depths = [3, 5, 7]
best_gini = -1.0
best_depth = None

print("Tuning Custom DecisionTreeClassifier:")
print("-" * 35)

for depth in max_depths:
    model = DecisionTreeClassifier(max_depth=depth)
    model.fit(X_train_processed, y_train)

    y_proba = model.predict_proba(X_valid_processed)[:, 1]

    try:
        auc = roc_auc_score(y_valid, y_proba)
        gini_score = 2 * auc - 1
    except ValueError as e:
        gini_score = -2.0
        print(f"Depth {depth}: Error calculating AUC/Gini ({e})")
        continue

    print(f"Depth {depth:<2}: Gini = {gini_score:.4f} (AUC = {auc:.4f})")

    if gini_score > best_gini:
        best_gini = gini_score
        best_depth = depth

print("-" * 35)
print(f"**Best Validation Gini Score: {best_gini:.4f} at Max Depth: {best_depth}**")

if best_gini >= 0.1:
    print("**TASK 3 SUCCESS:** Gini score target of 0.1 achieved!")
else:
    print("TASK 3 FAILURE: Gini score target not met. Re-check feature engineering.")

Tuning Custom DecisionTreeClassifier:
-----------------------------------
Depth 3 : Gini = 0.1742 (AUC = 0.5871)
Depth 5 : Gini = 0.1637 (AUC = 0.5818)
Depth 7 : Gini = 0.1540 (AUC = 0.5770)
-----------------------------------
**Best Validation Gini Score: 0.1742 at Max Depth: 3**
**TASK 3 SUCCESS:** Gini score target of 0.1 achieved!


## Task 4


>  Use sklearn's DecisionTreeClassifier and check its performance on the validation dataset. Is it better than your module? If so, why?



In [20]:
from sklearn.tree import DecisionTreeClassifier as SklearnDTC
from sklearn.metrics import roc_auc_score

optimal_depth = 5

training the scikit learn model

In [21]:
sklearn_model = SklearnDTC(max_depth=optimal_depth, criterion='gini', random_state=42)
sklearn_model.fit(X_train_processed, y_train)

DecisionTreeClassifier(max_depth=5, random_state=42)

predicting probabilities

In [22]:
y_proba_sklearn = sklearn_model.predict_proba(X_valid_processed)[:, 1]

calculating the gini and auc score

In [23]:
auc_sklearn = roc_auc_score(y_valid, y_proba_sklearn)
gini_sklearn = 2 * auc_sklearn - 1

the results

In [24]:
print(f"Scikit-learn Gini Score (Depth {optimal_depth}): {gini_sklearn:.4f}")

Scikit-learn Gini Score (Depth 5): 0.2484


## Task 5




> Implement the RandomForestClassifier and check its performance. You have to improve the result of a single tree and get at least 0.15 Gini score on the validation dataset. Be able to set a fixed random seed.



I am implementing randomforestclassifier and im using bagging and feature randomness

In [25]:
class RandomForestClassifier:
    def __init__(self, n_estimators=100, max_depth=None, max_features='sqrt', random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)

        N, D = X.shape
        if self.max_features == 'sqrt':
            self.n_features_per_split = int(np.sqrt(D))
        else:
            self.n_features_per_split = D

        self.trees = []
        for i in range(self.n_estimators):
            idx = np.random.randint(0, N, size=N)
            Xb = X[idx]
            yb = y[idx]

            tree = DecisionTreeClassifier(max_depth=self.max_depth, extra_randomization=True)
            tree.fit(Xb, yb)
            self.trees.append(tree)

    def predict_proba(self, X):
        probs = np.mean([t.predict_proba(X) for t in self.trees], axis=0)
        return probs

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)


now tuning and performance verification

so i need to tune the random forests hyperparameters, primarily n_estimators and max_depth, to maximize the gini score and hit the >= 0.15 target

In [26]:
N_ESTIMATORS = [100]
MAX_DEPTHS = [3] # Changed to a list

best_rf_gini = -1.0
best_params = {}
RANDOM_SEED = 42

print("Tuning Custom RandomForestClassifier:")
print("-" * 35)

for n_est in N_ESTIMATORS:
    for depth in MAX_DEPTHS:

        rf_model = RandomForestClassifier(
            n_estimators=n_est,
            max_depth=depth,
            random_state=RANDOM_SEED
        )
        rf_model.fit(X_train_processed, y_train)
        y_proba = rf_model.predict_proba(X_valid_processed)[:, 1]

        try:
            auc = roc_auc_score(y_valid, y_proba)
            gini_score = 2 * auc - 1
        except ValueError:
            gini_score = -2.0
            continue

        print(f"E={n_est:<3}, D={depth:<2}: Gini = {gini_score:.4f}")

        if gini_score > best_rf_gini:
            best_rf_gini = gini_score
            best_params = {'n_estimators': n_est, 'max_depth': depth}

print("-" * 35)
print(f"**Best Validation Gini Score: {best_rf_gini:.4f} with {best_params['n_estimators']} estimators and depth {best_params['max_depth']}**")

if best_rf_gini >= 0.15:
    print("**TASK 5 SUCCESS:** Gini score target of 0.15 achieved!")
else:
    print("TASK 5 FAILURE: Gini score target not met. Try increasing n_estimators or re-evaluating feature engineering.")

Tuning Custom RandomForestClassifier:
-----------------------------------
E=100, D=3 : Gini = 0.2442
-----------------------------------
**Best Validation Gini Score: 0.2442 with 100 estimators and depth 3**
**TASK 5 SUCCESS:** Gini score target of 0.15 achieved!


## Task 6





> Use your DecisionTree design class for GBDT classifier. This class must have max_depth, number_of_trees and max_features attributes. You must compute the gradient of the binary cross-entropy loss function and implement incremental learning: train the next tree using the results of the previous trees.



to implement a custom gbdt classifier, i need to leverage my existing decisiontreeregressor and adapt it to the classification task by using the negative gradient of the binary cross-entropy loss as target for each sequential tree

so i am taking that function from the task 2 and implemeting a new class from it

In [27]:
import numpy as np

class GBDTClassifier:
    """
    Лёгкий и быстрый GBDT на основе DecisionTreeRegressor.
    Использует бинарную логистическую потерю (logloss).
    """

    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1, subsample=1.0, colsample=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate

        self.subsample = subsample          # Stochastic gradient boosting
        self.colsample = colsample          # Random feature sampling

        self.random_state = random_state
        self.trees = []
        self.feature_subsets = []           # какие фичи использовало дерево
        self.initial_prediction = None

    # ---------------------------------------------------------
    # Utility functions
    # ---------------------------------------------------------

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def _compute_residuals(self, y, F):
        """
        Псевдо-градент бинарной кросс-энтропии: y - p
        """
        p = self._sigmoid(F)
        return y - p

    def _sample_rows(self, X, y):
        """
        Subsample rows (stochastic gradient boosting)
        """
        n = len(X)
        size = int(n * self.subsample)

        idx = np.random.choice(n, size=size, replace=False)
        return X[idx], y[idx]

    def _sample_columns(self, X):
        """
        Feature subsampling per tree
        """
        D = X.shape[1]
        k = int(D * self.colsample)

        cols = np.random.choice(D, k, replace=False)
        return cols

    # ---------------------------------------------------------
    # Fit
    # ---------------------------------------------------------

    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)

        N, D = X.shape
        avg = np.mean(y)

        # Инициализация F0 = log(p/(1-p))
        avg = np.clip(avg, 1e-6, 1-1e-6)
        self.initial_prediction = np.log(avg / (1 - avg))

        F = np.full(N, self.initial_prediction, dtype=float)
        self.trees = []
        self.feature_subsets = []

        for m in range(self.n_estimators):

            # 1. Compute gradients
            residuals = self._compute_residuals(y, F)

            # 2. Stochastic sampling
            if self.subsample < 1.0:
                X_m, r_m = self._sample_rows(X, residuals)
            else:
                X_m, r_m = X, residuals

            # 3. Column subsampling
            if self.colsample < 1.0:
                cols = self._sample_columns(X)
            else:
                cols = np.arange(D)

            self.feature_subsets.append(cols)

            X_sub = X_m[:, cols]

            # 4. Train regression tree on residuals
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X_sub, r_m)
            self.trees.append(tree)

            # 5. Update model
            F += self.learning_rate * tree.predict(X[:, cols])

    # ---------------------------------------------------------
    # Predict
    # ---------------------------------------------------------

    def predict_proba(self, X):
        N = len(X)
        F = np.full(N, self.initial_prediction, dtype=float)

        for tree, cols in zip(self.trees, self.feature_subsets):
            F += self.learning_rate * tree.predict(X[:, cols])

        p = self._sigmoid(F)
        return np.vstack([1 - p, p]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


so now verification and tuning, im gonna tune the learning_rate and n_estimators

In [28]:
N_ESTIMATORS = 100
MAX_DEPTH = 3
LEARNING_RATE = 0.1
gbdt_model = GBDTClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE
)
gbdt_model.fit(X_train_processed, y_train)

y_proba_gbdt = gbdt_model.predict_proba(X_valid_processed)[:, 1]
auc_gbdt = roc_auc_score(y_valid, y_proba_gbdt)
gini_gbdt = 2 * auc_gbdt - 1

print(f"\nGBDT Gini Score (E={N_ESTIMATORS}, D={MAX_DEPTH}, LR={LEARNING_RATE}): {gini_gbdt:.4f}")


GBDT Gini Score (E=100, D=3, LR=0.1): 0.3954


## Task 7



> Use LightGBM, Catboost, and XGBoost for fitting on a training set and prediction on a validation set. Review the documentation of the libraries and fine-tune the algorithms for the task. Note key differences between each implementation. Analyze special features of each algorithm (how does "categorical feature" work in Catboost, what is DART mode in XGBoost)? Which GBDT model gives the best result? Can you explain why?



Model Fitting and Tuning

all models will be tuned based on the AUC/Gini score on the validation set. i start with a small grid search, focusing on boosting rounds (n_estimators), learning rate, and tree depth.

sklearn

In [29]:
class GBDT_sklearn_like:
    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1,
                 subsample=1.0, colsample=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample = colsample
        self.random_state = random_state
        self.trees = []
        self.feature_subsets = []
        self.init_value = None

    def _sigmoid(self, x): return 1/(1+np.exp(-x))

    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)
        N, D = X.shape
        p = np.clip(np.mean(y),1e-6,1-1e-6)
        self.init_value = np.log(p/(1-p))
        F = np.full(N, self.init_value)
        self.trees = []
        self.feature_subsets = []
        for m in range(self.n_estimators):
            r = y - self._sigmoid(F)
            if self.subsample < 1:
                idx = np.random.choice(N, int(N*self.subsample), replace=False)
                X_m, r_m = X[idx], r[idx]
            else:
                X_m, r_m = X, r
            if self.colsample < 1:
                cols = np.random.choice(D, max(1,int(D*self.colsample)), replace=False)
            else:
                cols = np.arange(D)
            self.feature_subsets.append(cols)
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X_m[:, cols], r_m)
            self.trees.append(tree)
            F += self.learning_rate * tree.predict(X[:, cols])

    def predict_proba(self, X):
        N = len(X)
        F = np.full(N, self.init_value)
        for tree, cols in zip(self.trees, self.feature_subsets):
            F += self.learning_rate * tree.predict(X[:, cols])
        p = self._sigmoid(F)
        return np.vstack([1-p,p]).T

    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= 0.5).astype(int)


xgboost

In [30]:
class GBDT_xgboost_like:
    def __init__(self, n_estimators=100, max_depth=3, learning_rate=0.1,
                 subsample=1.0, colsample=1.0, reg_lambda=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample = colsample
        self.reg_lambda = reg_lambda
        self.random_state = random_state
        self.trees = []
        self.feature_subsets = []
        self.init_value = None

    def _sigmoid(self,x): return 1/(1+np.exp(-x))

    def fit(self, X, y):
        if self.random_state is not None:
            np.random.seed(self.random_state)
        N, D = X.shape
        p = np.clip(np.mean(y),1e-6,1-1e-6)
        self.init_value = np.log(p/(1-p))
        F = np.full(N, self.init_value)
        self.trees = []
        self.feature_subsets = []
        for m in range(self.n_estimators):
            p_pred = self._sigmoid(F)
            grad = y - p_pred
            hess = p_pred*(1-p_pred)
            if self.subsample < 1:
                idx = np.random.choice(N, int(N*self.subsample), replace=False)
                X_m, grad_m, hess_m = X[idx], grad[idx], hess[idx]
            else:
                X_m, grad_m, hess_m = X, grad, hess
            if self.colsample < 1:
                cols = np.random.choice(D, max(1,int(D*self.colsample)), replace=False)
            else:
                cols = np.arange(D)
            self.feature_subsets.append(cols)
            pseudo = grad_m/(hess_m+self.reg_lambda)
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X_m[:, cols], pseudo)
            self.trees.append(tree)
            F += self.learning_rate * tree.predict(X[:, cols])

    def predict_proba(self, X):
        F = np.full(len(X), self.init_value)
        for tree, cols in zip(self.trees, self.feature_subsets):
            F += self.learning_rate * tree.predict(X[:, cols])
        p = self._sigmoid(F)
        return np.vstack([1-p,p]).T

    def predict(self, X):
        return (self.predict_proba(X)[:,1] >= 0.5).astype(int)


lightgbm

In [31]:
class GBDT_lightgbm_like:
    def __init__(self, n_estimators=100, max_leaves=31, learning_rate=0.1,
                 subsample=1.0, colsample=1.0, random_state=None):
        self.n_estimators = n_estimators
        self.max_leaves = max_leaves
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample = colsample
        self.random_state = random_state
        self.trees=[]
        self.feature_subsets=[]
        self.init_value=None

    def _sigmoid(self,x): return 1/(1+np.exp(-x))

    def fit(self,X,y):
        if self.random_state is not None:
            np.random.seed(self.random_state)
        N,D=X.shape
        p=np.clip(np.mean(y),1e-6,1-1e-6)
        self.init_value=np.log(p/(1-p))
        F=np.full(N,self.init_value)
        self.trees=[]
        self.feature_subsets=[]
        for m in range(self.n_estimators):
            r=y-self._sigmoid(F)
            if self.subsample<1:
                idx=np.random.choice(N,int(N*self.subsample),replace=False)
                X_m,r_m=X[idx],r[idx]
            else:
                X_m,r_m=X,r
            if self.colsample<1:
                cols=np.random.choice(D,max(1,int(D*self.colsample)),replace=False)
            else:
                cols=np.arange(D)
            self.feature_subsets.append(cols)
            tree=DecisionTreeRegressor(max_depth=7)
            tree.fit(X_m[:,cols],r_m)
            self.trees.append(tree)
            F+=self.learning_rate*tree.predict(X[:,cols])

    def predict_proba(self,X):
        F=np.full(len(X),self.init_value)
        for tree,cols in zip(self.trees,self.feature_subsets):
            F+=self.learning_rate*tree.predict(X[:,cols])
        p=self._sigmoid(F)
        return np.vstack([1-p,p]).T

    def predict(self,X):
        return (self.predict_proba(X)[:,1]>=0.5).astype(int)


CatBoostGBM

In [32]:
import pandas as pd

class GBDT_catboost_like:
    def __init__(self,n_estimators=100,max_depth=3,learning_rate=0.1,
                 subsample=1.0,colsample=1.0,random_state=None,
                 cat_features=None,prior=0.5):
        self.n_estimators=n_estimators
        self.max_depth=max_depth
        self.learning_rate=learning_rate
        self.subsample=subsample
        self.colsample=colsample
        self.random_state=random_state
        self.cat_features=cat_features or []
        self.prior=prior
        self.trees=[]
        self.feature_subsets=[]
        self.init_value=None
        self.cat_stats={}

    def _sigmoid(self,x): return 1/(1+np.exp(-x))

    def _ordered_encode(self,df,y,seed):
        N=len(df)
        rng=np.random.RandomState(seed)
        perm=np.arange(N)
        rng.shuffle(perm)
        inv=np.argsort(perm)
        out={}
        for col in self.cat_features:
            s={}
            c={}
            enc=np.zeros(N)
            for idx in perm:
                key=df.loc[idx,col]
                a=s.get(key,0); b=c.get(key,0)
                enc[idx]=(a+self.prior)/(b+1)
                s[key]=a+y[idx]
                c[key]=b+1
            enc=enc[inv]
            out[col]=enc
            df[col]=enc
        return df,out

    def fit(self,X_df,y):
        if isinstance(X_df,np.ndarray):
            df=pd.DataFrame(X_df)
        else:
            df=X_df.reset_index(drop=True).copy()
        if self.random_state is not None:
            np.random.seed(self.random_state)
        N,D=df.shape
        p=np.clip(np.mean(y),1e-6,1-1e-6)
        self.init_value=np.log(p/(1-p))
        if len(self.cat_features)>0:
            df,enc=self._ordered_encode(df,y,self.random_state or 0)
            self.cat_stats=enc
        X=df.values
        F=np.full(N,self.init_value)
        self.trees=[]
        self.feature_subsets=[]
        for m in range(self.n_estimators):
            r=y-self._sigmoid(F)
            if self.subsample<1:
                idx=np.random.choice(N,int(N*self.subsample),replace=False)
                X_m,r_m=X[idx],r[idx]
            else:
                X_m,r_m=X,r
            if self.colsample<1:
                cols=np.random.choice(D,max(1,int(D*self.colsample)),replace=False)
            else:
                cols=np.arange(D)
            self.feature_subsets.append(cols)
            tree=DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X_m[:,cols],r_m)
            self.trees.append(tree)
            F+=self.learning_rate*tree.predict(X[:,cols])

    def predict_proba(self,X_df):
        if isinstance(X_df,np.ndarray):
            X=X_df
        else:
            df=X_df.reset_index(drop=True).copy()
            for col,enc in self.cat_stats.items():
                df[col]=df[col].map(lambda x:enc.mean())
            X=df.values
        F=np.full(len(X),self.init_value)
        for tree,cols in zip(self.trees,self.feature_subsets):
            F+=self.learning_rate*tree.predict(X[:,cols])
        p=self._sigmoid(F)
        return np.vstack([1-p,p]).T

    def predict(self,X_df):
        return (self.predict_proba(X_df)[:,1]>=0.5).astype(int)


## Task 8



> Take the best model and estimate its performance on the test dataset: check the Gini values on all three datasets for your best model: training Gini, valid Gini, test Gini. Do you see a drop in performance when comparing the valid quality to the test quality? Is your model overfitting or not? Explain.



In [33]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

def evaluate_model(model, X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    accs, f1s, aucs = [], [], []
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:,1]
        accs.append(accuracy_score(y_test, y_pred))
        f1s.append(f1_score(y_test, y_pred))
        aucs.append(roc_auc_score(y_test, y_proba))
    return np.mean(accs), np.mean(f1s), np.mean(aucs)

def task8_run_all(X, y):
    models = {
        "sklearn_like": GBDT_sklearn_like(),
        "xgboost_like": GBDT_xgboost_like(),
        "lightgbm_like": GBDT_lightgbm_like(),
        "catboost_like": GBDT_catboost_like(cat_features=[]),
        "random_forest": RandomForestClassifier(),
        "decision_tree": DecisionTreeClassifier()
    }

    results = {}
    for name, model in models.items():
        acc, f1, auc = evaluate_model(model, X, y)
        results[name] = (acc, f1, auc)

    return results


summary

In [34]:
def print_summary(results):
    print("MODEL SUMMARY")
    print("-" * 40)
    for name, (acc, f1, auc) in results.items():
        print(f"{name:20s}  ACC={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}")



In [39]:
X = np.vstack([X_train_processed, X_valid_processed, X_test_processed])
y = np.concatenate([y_train, y_valid, y_test])


## Task 9 *



> Implement the ExtraTreesClassifier and check its performance. You must improve the result of a single tree and obtain a Gini score of at least 0.12 on the validation dataset.



In [37]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def task9_final_training(X_train, y_train, X_test, y_test, model_class):
    model = model_class()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)

    return model, acc, f1, auc


In [40]:
best_model_class = GBDT_xgboost_like
model, acc, f1, auc = task9_final_training(X_train_processed, y_train, X_test_processed, y_test, best_model_class)

In [41]:
print("Summary: ", best_model_class.__name__)


Summary:  GBDT_xgboost_like
----------------------------------------
